In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import style
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix, roc_curve, accuracy_score
from sklearn.preprocessing import StandardScaler
from imblearn.combine import SMOTEENN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE

In [ ]:
df = pd.read_excel("Data/datos_limpios.xlsx")
df

In [ ]:
print("Número de observaciones por clase")
print(df['Impago'].value_counts())
print("")

print("Porcentaje de observaciones por clase")
print(100 * df['Impago'].value_counts(normalize=True))


El 88% de los correos no son spam y el 11% sí lo son. Un modelo de clasificación que sea útil debe de ser capaz de predecir correctamente un porcentaje de observaciones por encima del porcentaje de la clase mayoritaria. En este caso, el umbral de referencia que se tiene que superar es del 88%.

In [ ]:

X = df[['Edad', 'Ingresos', 'Monto_Inicial', 'Scoring_Crediticio',
       'Num_Creditos', 'Ratio_Interes',
       'Ratio_Deuda_Ingresos','Posesion_Hipoteca',
       'Personas_Cargo', 'Fiador', ]] 
y = df['Impago']

X_train, X_test, y_train, y_test = train_test_split(
                                        X, y.values,
                                        train_size   = 0.7,
                                        random_state = 42,
                                        shuffle      = True,
                                        stratify = y)

In [ ]:
# Escalado de variables

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)

In [ ]:
# 2. Undersampling: Reducimos los buenos pagadores (clase 0)
undersampler = RandomUnderSampler(sampling_strategy={0: 30000}, random_state=42)
X_train_under, y_train_under = undersampler.fit_resample(X_train_scaled, y_train)

# 3. SMOTE: Subimos los morosos (clase 1)
smote = SMOTE(sampling_strategy={1: 30000}, random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_under, y_train_under)

print(f"Distribución para entrenar -> 0: {sum(y_train_resampled==0)} | 1: {sum(y_train_resampled==1)}")

In [ ]:
# smote_enn = SMOTEENN(random_state=42)
# X_train_resampled, y_train_resampled = smote_enn.fit_resample(X_train_scaled, y_train)

In [ ]:
modelo_log_sklearn = LogisticRegression(random_state=42, max_iter=1000)
modelo_log_sklearn.fit(X_train_resampled, y_train_resampled)

In [ ]:
# Información del modelo
print("Intercept:", modelo_log_sklearn.intercept_)
print("Coeficiente:", list(zip(X_train_resampled.columns, modelo_log_sklearn.coef_.flatten(), )))
print("Accuracy de entrenamiento:", modelo_log_sklearn.score(X_train_resampled, y_train_resampled))

In [ ]:
# predict_proba()
X_test_scaled = scaler.transform(X_test)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

predicciones = modelo_log_sklearn.predict_proba(X = X_test_scaled)
predicciones = pd.DataFrame(predicciones, columns = modelo_log_sklearn.classes_)
predicciones.head(5)

In [ ]:
# Creación del modelo utilizando matrices como sklearn

X_train_scaled = sm.add_constant(X_train_scaled, prepend=True)
modelo_log_statsm = sm.Logit(endog= y_train, exog = X_train_scaled)
modelo_log_statsm = modelo_log_statsm.fit()
print(modelo_log_statsm.summary())

In [ ]:
intervalos_ci = modelo_log_statsm.conf_int(alpha=0.05).rename(columns={0: "2.5%", 1: "97.5%"})

intervalos_ci[:4]

In [ ]:
# Debemos añadir una columna de constante al conjunto de test 

X_test_scaled = sm.add_constant(X_test_scaled, has_constant='add')

predicciones = modelo_log_statsm.predict(exog = X_test_scaled)
predicciones[:4]

In [ ]:
# # Hallamos la clasificación predicha estableciendo un umbral de probabilidad 0.5

# clasificacion = np.where(predicciones<0.5, 0, 1)
# clasificacion[:4]

In [ ]:
# Quitamos la columna 'const' (añadida por statsmodels) para que sklearn pueda predecir
X_test_sklearn = X_test_scaled.drop(columns=['const'], errors='ignore')

# Obtenemos solo la columna de probabilidades de la clase 1 (Impago)
probabilidades_impago = modelo_log_sklearn.predict_proba(X_test_sklearn)[:, 1]

# Definimos el umbral de decisión (puedes ir bajándolo, ej: 0.20, 0.15...)
umbral_decision = 0.50

# Hallamos la clasificación predicha estableciendo el umbral
clasificacion = np.where(probabilidades_impago < umbral_decision, 0, 1)

print(clasificacion[:4])

### Métricas de rendimiento del modelo

- Accuracy

- precision

- sensibilidad (recall)

- f1-score

- Matriz de confusión

- Curva precisión/sensibilidad

- curva ROC-AUC

In [ ]:
# Accuracy / Aciertos: 
""" 
Mide la proporción de predicciones correctas sobre el total de observaciones evaluadas
"""

from sklearn.metrics import accuracy_score

acierto = accuracy_score(y_test, clasificacion)
error = 1 - acierto
print("Acierto:", round(acierto*100, 2), "%")
print("Error:", round(error*100, 2), "%")

Acierto: 63.4 %
Error: 36.6 %

In [ ]:
# Precisión, Recall, f1-score
""" 
Precision: Cada vez que el modelo predijo una clase, acerto ese porcentaje
Recall: De todos los datos que eran de esa clase, detectó ese porcentaje
F1-score: El valor será alto si precisión y sensibilidad son similares, útil para comparar clasificadores.
"""



from sklearn.metrics import classification_report

print(classification_report(y_test, clasificacion))


 0       0.94      0.63      0.75     13530
1        0.19      0.67      0.30      1772

In [ ]:
# Matriz de confusión

from sklearn.metrics import confusion_matrix

c_matrix = confusion_matrix(y_test, clasificacion)
print(c_matrix)

sns.heatmap(c_matrix,annot=True, fmt="d", cmap="Blues")

In [ ]:
## Curva precisión/recall

""" 
Cómo varía la precisión y sensibilidad respecto a diferentes umbrales de probabilidad
"""
from sklearn.metrics import precision_recall_curve

# Primero creamos una tabla para ver como varía la precisión y recall en función del valor umbral

prec, rec, thresholds = precision_recall_curve(y_test, predicciones) # necesitamos la probabilidad de la clase 1
pd.DataFrame({"prec":prec[:-1], "rec":rec[:-1], "threshold":thresholds})[80:100]


In [ ]:
# Creamos la curva

plt.plot(rec, prec)

plt.xlabel("Recall")
plt.ylabel("Precision")

plt.title("PR Curve")

In [ ]:
import matplotlib.pyplot as plt

plt.plot(fpr, tpr)

plt.xlabel("Ratio de falsos positivos")
plt.ylabel("Ratio de verdaderos positivos")
plt.title("Curva ROC")